# Deduplication walkthrough

Hands-on tour of the two dedup stages exposed by `corpus_prep.dedup`:

| Stage | Function | Rationale |
|---|---|---|
| Pre-dedup | `dedup_files()` | Drop byte-identical files (SHA-256) before the parsing stage — saves CPU on duplicate ingestion. |
| Post-dedup | `dedup_documents()` | Drop near-duplicate documents (MinHash LSH, Jaccard >= 0.8) after parsing — catches paraphrased text and OCR variants. |

**Why two stages?** Files arriving in an ingestion drop frequently include exact copies under different names (e.g. `diario_001.pdf` and `diario_001 (1).pdf`). Hashing those is cheap and we can short-circuit the entire pipeline. Once parsing has produced text, near-duplicates from different sources (e.g. the same press release scraped from two outlets) only become detectable via content similarity.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

## 1. Pre-dedup: byte-identical files

We create a temporary directory with three files:
- `a.txt` and `b.txt` have **identical content** but different names.
- `c.txt` is unique.

`dedup_files` keeps the first occurrence of each unique SHA-256 and drops subsequent duplicates.

In [ ]:
import tempfile
from pathlib import Path

from corpus_prep.dedup import dedup_files, file_sha256

tmp = Path(tempfile.mkdtemp())
(tmp / "a.txt").write_text("Conteudo identico para teste de dedup.")
(tmp / "b.txt").write_text("Conteudo identico para teste de dedup.")
(tmp / "c.txt").write_text("Outro conteudo, totalmente distinto.")

files = sorted(tmp.glob("*.txt"))
for f in files:
    print(f"{f.name:8s}  sha256={file_sha256(f)[:16]}...")

kept, dropped = dedup_files(files)
print(f"\nkept:    {[f.name for f in kept]}")
print(f"dropped: {[f.name for f in dropped]}")

## 2. Post-dedup: near-duplicate documents (MinHash LSH)

We build three Documents:
- `doc_a` — the canonical text.
- `doc_b` — almost identical to `doc_a` (one word changed).
- `doc_c` — unrelated content.

MinHash LSH groups `doc_a` and `doc_b` by Jaccard similarity above the threshold; `doc_c` stays.

In [ ]:
from datetime import datetime, timezone

from corpus_prep.dedup import dedup_documents, make_minhash
from corpus_prep.schemas import Document
from corpus_prep.utils.ids import uuid7


def doc(text: str) -> Document:
    return Document(
        id=uuid7(),
        text=text,
        source_path=Path("x.txt"),
        mime="text/plain",
        parser="plaintext",
        sha256="a" * 64,
        char_count=len(text),
        extracted_at=datetime.now(timezone.utc),
    )


TEXT_A = (
    "Large language models trained on web-scale corpora exhibit emergent "
    "behaviors when scaled past a certain parameter count. Researchers at "
    "leading labs have demonstrated that few-shot reasoning, code synthesis, "
    "and multi-step planning all improve smoothly with scale until they "
    "abruptly cross usability thresholds."
)
TEXT_B_NEAR = TEXT_A.replace("abruptly", "suddenly")  # one-word edit
TEXT_C_UNRELATED = (
    "Brazilian cuisine reflects a deep blend of indigenous, African, and "
    "European influences. Regional dishes range from feijoada in the southeast "
    "to tacaca and acai bowls in the Amazon basin."
)

doc_a = doc(TEXT_A)
doc_b = doc(TEXT_B_NEAR)
doc_c = doc(TEXT_C_UNRELATED)

### Pairwise Jaccard (sanity check)

Before plugging into LSH, compute the raw MinHash Jaccard estimates so the LSH decisions are interpretable.

In [ ]:
ma = make_minhash(TEXT_A)
mb = make_minhash(TEXT_B_NEAR)
mc = make_minhash(TEXT_C_UNRELATED)

print(f"jaccard(a, b near-dup) = {ma.jaccard(mb):.3f}")
print(f"jaccard(a, c unrelated) = {ma.jaccard(mc):.3f}")
print(f"jaccard(b, c unrelated) = {mb.jaccard(mc):.3f}")

### Run dedup at the default threshold (0.8)

In [ ]:
kept, removed = dedup_documents([doc_a, doc_b, doc_c], threshold=0.8)

print("Default threshold = 0.8")
print(f"kept:    {len(kept)} document(s)")
for d in kept:
    print(f"  - {d.id} :: {d.text[:60]}...")
print(f"removed: {len(removed)} document(s)")
for rid in removed:
    print(f"  - {rid}")

## 3. Threshold tuning

Lower thresholds are aggressive (collapse more); higher thresholds are conservative (keep more). The default 0.8 follows the FineWeb / datatrove convention.

We sweep across thresholds to visualize the effect on the same three documents.

In [ ]:
for t in [0.5, 0.7, 0.8, 0.9, 0.95]:
    kept, removed = dedup_documents([doc_a, doc_b, doc_c], threshold=t)
    print(f"threshold={t}: kept={len(kept)}, removed={len(removed)}")

## Takeaways

- **Pre-dedup is essentially free** — SHA-256 over a 100 MB file finishes in well under a second on commodity hardware. Always run it before parsing.
- **Post-dedup catches what hashing misses** — copy-paste with light edits, OCR variants of the same page, paraphrased press releases.
- **0.8 is a sane default** but worth tuning per corpus. Heavily templated content (forms, court docs) often benefits from a lower threshold (0.7); high-quality editorial text can stay at 0.85.
- The implementation uses **datasketch** for the MinHash and LSH primitives. The text-dedup project wraps the same primitives for batch CLI usage.